In [1]:
import os
import pandas as pd
import ast

from src.utils import get_base_dir, load_from_pickle

In [2]:
region = "Basilicata" #"Basilicata" #
region_str = region.lower() # "emilia_romagna" # #
base_dir = os.path.join(get_base_dir(), region)

In [3]:
# prendo il mapping delle review che mi serve per filtrare llm_output
mapping_R = load_from_pickle(os.path.join(base_dir, "subset", "mapping.pkl"))["review"]
reviews_ok = list(mapping_R.values())


In [4]:
mapping_U = load_from_pickle(os.path.join(base_dir, "subset", "mapping.pkl"))["user"]
users_ok = list(mapping_U.values())
users_ok[:10]

[56, 464, 624, 735, 1016, 1196, 1339, 1385, 1464, 1604]

In [5]:
#prendo file con llm_output
df_gt = pd.read_parquet(os.path.join(base_dir, "raw_data", f"review_{region_str}_filtered_by_user_output_llm.parquet"))
df_gt.shape

(53468, 16)

In [6]:
#prendo subset delle colonne che mi interessano: id della review, testo della review e output dell'llm (e applico mapping, importante, ero in raw_data)

df_gt = df_gt[["unique_id", "review_text", "llm_output"]].copy()
df_gt["unique_id"] = df_gt["unique_id"].apply(lambda x: x - 1)
df_gt.head()

,unique_id,review_text,llm_output
0,19,None,None
1,46,None,None
2,104,None,None
3,261,Bello...peccato che duri poco,"{'Animal Beach': None, 'Animal Lover': 0.0, 'A..."
4,277,Da provare,"{'Animal Beach': None, 'Animal Lover': 0.0, 'A..."


In [7]:
#filtro solo le review di interesse

df_gt = df_gt[~df_gt["llm_output"].isna()]
df_gt_sub = df_gt[df_gt["unique_id"].isin(reviews_ok)]
df_gt_sub.shape

(36157, 3)

In [8]:
# aggiornamento review ok

reviews_ok_new = df_gt_sub["unique_id"].tolist()
len(reviews_ok_new)

36157

In [9]:
# prendiamo poi della review e categoria del poi.

#review - poi
dfRP = pd.read_parquet(os.path.join(base_dir, "original_data", "edges", "review_is_about_poi.parquet"))
dfRP_ok = dfRP[dfRP["unique_id_review"].isin(reviews_ok_new)]
dict_RP = dict(zip(dfRP_ok["unique_id_review"], dfRP_ok["unique_id_poi"]))

#poi
pois_ok = dfRP_ok["unique_id_poi"].drop_duplicates().tolist()
dfP = pd.read_parquet(os.path.join(base_dir, "original_data", "nodes", "poi.parquet"))
dfP_sub = dfP[["unique_id", "title", "type", "category"]].copy()
dfP_ok = dfP_sub[dfP_sub["unique_id"].isin(pois_ok)]
dict_Pname = dict(zip(dfP_ok["unique_id"], dfP_ok["title"]))
dict_Ptype = dict(zip(dfP_ok["unique_id"], dfP_ok["type"]))
dict_Pcategory = dict(zip(dfP_ok["unique_id"], dfP_ok["category"]))

#update
df_gt_sub["poi_name"] = df_gt_sub["unique_id"].map(dict_RP)
df_gt_sub["poi_name"] = df_gt_sub["poi_name"].map(dict_Pname)

df_gt_sub["poi_type"] = df_gt_sub["unique_id"].map(dict_RP)
df_gt_sub["poi_type"] = df_gt_sub["poi_type"].map(dict_Ptype)

df_gt_sub["poi_category"] = df_gt_sub["unique_id"].map(dict_RP)
df_gt_sub["poi_category"] = df_gt_sub["poi_category"].map(dict_Pcategory)

df_gt_sub.head()

/tmp/ipykernel_3793684/3633181513.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gt_sub["poi_name"] = df_gt_sub["unique_id"].map(dict_RP)
/tmp/ipykernel_3793684/3633181513.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gt_sub["poi_name"] = df_gt_sub["poi_name"].map(dict_Pname)
/tmp/ipykernel_3793684/3633181513.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the doc

,unique_id,review_text,llm_output,poi_name,poi_type,poi_category
3,261,Bello...peccato che duri poco,"{'Animal Beach': None, 'Animal Lover': 0.0, 'A...",Volo dell'Aquila San Costantino Albanese,attractions,Attrazione turistica
4,277,Da provare,"{'Animal Beach': None, 'Animal Lover': 0.0, 'A...",Volo dell'Aquila San Costantino Albanese,attractions,Attrazione turistica
5,304,Personale adorabile e dedicato che ha aiutato ...,"{'Animal Beach': None, 'Animal Lover': 0.0, 'A...",Volo dell'Aquila San Costantino Albanese,attractions,Attrazione turistica
10,533,La collina Materana è una zona collinare inter...,"{'Animal Beach': None, 'Animal Lover': 0.1, 'A...",Collina Materana,attractions,Riserva naturale
11,554,"Locale accogliente, ben arredato, dove si poss...","{'Animal Beach': None, 'Animal Lover': 0.0, 'A...",La stradella,restaurants,Italiana


In [10]:
#Otteniamo una colonna per ogni classe/profilo utente

def safe_parse(val):
    if isinstance(val, str):
        return ast.literal_eval(val)
    return val

df_gt_sub["parsed_output"] = df_gt_sub["llm_output"].apply(safe_parse)
df_expanded = pd.json_normalize(df_gt_sub["parsed_output"])
df_expanded.drop(["Animal Beach", "Animal Spiaggia", "spiegazione"], axis=1, inplace=True)

translation_map = {
    'Curioso': 'curious',
    'Esploratore': 'explorer',
    'Colto': 'cultured',
    'Spirito Libero': 'free_spirit',
    'Romantico': 'romantic',
    'Famiglia': 'family_oriented',
    'Animale Spiaggia': 'beach_person',
    'Pigro': 'lazy',
    'Animal Lover': 'animal_lover',
    'Viaggiatore Online': 'online_traveler',
    'Disorganizzato': 'disorganized',
    'Viaggiatore Verde': 'eco_traveler',
    'GenZ': 'genZ',
    'Nomadi Digitali': 'digital_nomad',
    'Viaggiatore Affari': 'business_traveler'
}

#df_expanded.rename(columns=translation_map, inplace=True)
df_final = pd.concat([
    df_gt_sub["unique_id"].reset_index(drop=True), 
    df_gt_sub["review_text"].reset_index(drop=True), 
    df_gt_sub["poi_name"].reset_index(drop=True),
    df_gt_sub["poi_type"].reset_index(drop=True),
    df_gt_sub["poi_category"].reset_index(drop=True),
    df_expanded.reset_index(drop=True)
], axis=1)
df_final["unique_id"].astype(int)

df_final.head()

/tmp/ipykernel_3793684/1970968657.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gt_sub["parsed_output"] = df_gt_sub["llm_output"].apply(safe_parse)


,unique_id,review_text,poi_name,poi_type,poi_category,Animal Lover,Animale Spiaggia,Colto,Curioso,Disorganizzato,Esploratore,Famiglia,GenZ,Nomadi Digitali,Pigro,Romantico,Spirito Libero,Viaggiatore Affari,Viaggiatore Online,Viaggiatore Verde
0,261,Bello...peccato che duri poco,Volo dell'Aquila San Costantino Albanese,attractions,Attrazione turistica,0.0,0.0,0.0,0.2,0.0,0.1,0.1,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.1
1,277,Da provare,Volo dell'Aquila San Costantino Albanese,attractions,Attrazione turistica,0.0,0.0,0.1,0.2,0.1,0.1,0.1,0.1,0.0,0.0,0.0,0.0,0.0,0.2,0.1
2,304,Personale adorabile e dedicato che ha aiutato ...,Volo dell'Aquila San Costantino Albanese,attractions,Attrazione turistica,0.0,0.0,0.1,0.2,0.1,0.3,0.6,0.2,0.0,0.0,0.0,0.1,0.0,0.2,0.1
3,533,La collina Materana è una zona collinare inter...,Collina Materana,attractions,Riserva naturale,0.1,0.0,0.1,0.2,0.1,0.7,0.2,0.3,0.1,0.0,0.1,0.3,0.0,0.2,0.5
4,554,"Locale accogliente, ben arredato, dove si poss...",La stradella,restaurants,Italiana,0.0,0.0,0.0,0.2,0.0,0.0,0.3,0.2,0.0,0.0,0.0,0.1,0.0,0.1,0.0


In [11]:
#prendiamo gli utenti che scrivono le review, e filtriamo il sottoinsieme ok

dfUR = pd.read_parquet(os.path.join(base_dir, "original_data", "edges", "user_wrote_review.parquet"))
print(dfUR.shape)
df_UR_ok = dfUR[dfUR["unique_id_review"].isin(reviews_ok_new)]
print(df_UR_ok.shape)

(1700367, 2)
(36157, 2)


In [12]:
# add user column and group by user

df_merged = df_final.merge(
    df_UR_ok,
    how='left',
    left_on='unique_id',
    right_on='unique_id_review'
)
df_merged.drop(columns=['unique_id_review', 'unique_id'], inplace=True)
#df_merged.head()

agg_dict = {
    col: 'mean' if df_merged[col].dtype in ['float64', 'int64'] else (lambda x: list(x))
    for col in df_merged.columns if 'id' not in col
}

df_grouped = df_merged.groupby("unique_id_user").agg(agg_dict).reset_index()

df_grouped.head()


,unique_id_user,review_text,poi_name,poi_type,poi_category,Animal Lover,Animale Spiaggia,Colto,Curioso,Disorganizzato,Esploratore,Famiglia,GenZ,Nomadi Digitali,Pigro,Romantico,Spirito Libero,Viaggiatore Affari,Viaggiatore Online,Viaggiatore Verde
0,56,"[Ottima pizza, Ottima pizza davvero buona e di...","[Il Buon Gusto, Pizzeria Punto Giusto...forno ...","[restaurants, restaurants, restaurants, restau...","[Pizza, Pizza, Pub, Fornaio, Pub, Fornaio, Piz...",0.000000,0.0,0.021918,0.208219,0.109589,0.008219,0.184932,0.230137,0.004110,0.013699,0.020548,0.108219,0.001370,0.154795,0.039726
1,464,[Personale gentilissimo mi dicono... mai entra...,[Museo Archeologico Nazionale della Basilicata...,"[attractions, restaurants, restaurants, restau...","[Museo archeologico, Gelato, Fast food, Pizza,...",0.004348,0.0,0.026087,0.121739,0.063043,0.013043,0.169565,0.150000,0.000000,0.010870,0.008696,0.050000,0.013043,0.117391,0.021739
2,624,"[Locale carino, ordinato ed arredato con cura....","[La Dolce Vita Ristorante Pizzeria Rivello, L'...","[restaurants, restaurants, restaurants, restau...","[Ristorante, Pizza, Pizza, Pizza, Pizza, Bar, ...",0.004545,0.0,0.040909,0.213636,0.068182,0.063636,0.286364,0.227273,0.009091,0.022727,0.134091,0.140909,0.000000,0.190909,0.077273
3,735,[Molto bello e tutto era perfetto.],[Dimora dell'Annunziata],[hotel],[Hotel],0.000000,0.0,0.000000,0.000000,0.100000,0.000000,0.200000,0.000000,0.000000,0.600000,0.300000,0.000000,0.000000,0.100000,0.000000
4,1016,"[Chiesa rupestre tipica di Matera, mantenuta i...","[Chiesa Rupestre di Santa Maria di Idris, L'An...","[attractions, restaurants, attractions, attrac...","[Chiesa cattolica, Panini, Attrazione turistic...",0.000000,0.0,0.620000,0.620000,0.140000,0.140000,0.340000,0.400000,0.080000,0.120000,0.160000,0.060000,0.000000,0.500000,0.220000


In [13]:
# Sort by length of the lists in 'text_review', descending
df_ordered = df_grouped.sort_values(
    by='review_text',
    key=lambda x: x.str.len(),  # get length of each list
    ascending=False
).reset_index(drop=True)

df_ordered.head(50)

,unique_id_user,review_text,poi_name,poi_type,poi_category,Animal Lover,Animale Spiaggia,Colto,Curioso,Disorganizzato,Esploratore,Famiglia,GenZ,Nomadi Digitali,Pigro,Romantico,Spirito Libero,Viaggiatore Affari,Viaggiatore Online,Viaggiatore Verde
0,488819,"[Bell'arredamento, buoni prodotti, personale c...","[Pizzeria I Due Sassi, Ohana, Scipipì Bistrot ...","[restaurants, restaurants, restaurants, restau...","[Pizza, Cucina hawaiana, Ristorante, Fornaio, ...",0.000629,0.000000,0.181761,0.325000,0.073742,0.035220,0.233648,0.234748,0.025314,0.034652,0.062264,0.122327,0.024528,0.240881,0.079717
1,455947,"[Un amico di una vita, sempre gentilissimo e c...","[Tabacchi e Caffetteria D'Anzi, Pane e Bontà, ...","[restaurants, restaurants, restaurants, restau...","[Bar, Fornaio, Pizza, Fornaio, Fornaio, Pizza,...",0.000714,0.000000,0.111786,0.293929,0.065000,0.031786,0.249643,0.207143,0.007500,0.020536,0.047143,0.110893,0.007679,0.165179,0.065000
2,156259,"[Buona la pizza e tutti i prodotti da forno, T...","[Pane e Bontà, Enoteca Non Solo Vino, La Sforn...","[restaurants, restaurants, restaurants, restau...","[Fornaio, Enoteca, Fornaio, Birreria, Pizza, B...",0.003750,0.002885,0.138542,0.280000,0.066250,0.040417,0.201042,0.203125,0.019375,0.013958,0.032500,0.083125,0.012917,0.176042,0.080417
3,401118,[Panificio storico è l'unico a Matera aperto d...,"[Pane e Bontà, Pizzeria One The Original, Caff...","[restaurants, restaurants, restaurants, restau...","[Fornaio, Pizza, Bar caffè, Pizza, Pizza, Forn...",0.003333,0.003738,0.179167,0.286250,0.065000,0.049167,0.238333,0.209167,0.013750,0.022083,0.053542,0.068125,0.004375,0.194167,0.088958
4,292137,"[Superiore, Molto raffinato, Superiore, Superi...","[Caffetteria, Tabacchi e..., Mezzamaro, Palazz...","[restaurants, restaurants, restaurants, restau...","[Tabaccheria, Italiana, Ristorante, Pizza, Bar...",0.003721,0.005495,0.089070,0.306977,0.084651,0.019767,0.200000,0.265116,0.025116,0.040465,0.103023,0.180000,0.018605,0.186977,0.065814
5,391966,"[Ottimo bar, Ottime le pizze, Coffi store ecce...","[Caffè di Pede | Deposito Bagagli, Pizza Deliv...","[restaurants, restaurants, restaurants, restau...","[Bar, Pizza, Caffetteria, Bar caffè, Pizza, Ba...",0.002463,0.004046,0.083498,0.248768,0.070936,0.013793,0.220443,0.204926,0.005172,0.034483,0.033498,0.088670,0.014286,0.162315,0.039901
6,349404,"[Ottima, Buon posto per trancio di pizza fatta...","[Caffetteria, Tabacchi e..., Pizzeria One The ...","[restaurants, restaurants, restaurants, restau...","[Tabaccheria, Pizza, Pizza, Bar caffè, Fornaio...",0.000000,0.000000,0.083000,0.242000,0.055000,0.026500,0.214250,0.191750,0.010000,0.028750,0.050000,0.081250,0.024500,0.148750,0.049500
7,356359,"[Niente di che, Più che una tabaccheria un luo...","[Pizza Alla Goccia, Tabaccheria Effepi F.lli L...","[restaurants, restaurants, restaurants, restau...","[Pizza, Bar, Bar caffè, Ristorante, Bar, Bed &...",0.000758,0.000000,0.343182,0.382323,0.071717,0.057576,0.215404,0.209848,0.017424,0.020960,0.058838,0.048990,0.009091,0.236616,0.117172
8,428519,[Ottimo l'hamburger! Buone le patatine. Vasta ...,"[Central Pub, Pummarola, Osteria Malatesta, Ca...","[restaurants, restaurants, restaurants, attrac...","[Pub, Pizza da asporto, Ristorante, Attrazione...",0.002717,0.028931,0.142663,0.273913,0.067935,0.045380,0.219565,0.204891,0.015761,0.073098,0.099728,0.116576,0.011413,0.176630,0.074185
9,569780,[Storica tabaccheria dal 1960...locale rinnova...,"[Tabacchi e Caffetteria D'Anzi, Caffeina, Pane...","[restaurants, restaurants, restaurants, restau...","[Bar, Caffetteria, Pizza, Bar, Fast food, Alim...",0.000610,0.000000,0.075305,0.211585,0.064024,0.014024,0.227744,0.167378,0.004268,0.031707,0.056098,0.072866,0.010976,0.140549,0.042683


In [42]:
len(df_ordered.iloc[49]["review_text"])

98

In [14]:
df_ordered.to_parquet(os.path.join(base_dir, "output_per_esperti.parquet"), index=False)

In [15]:
df = pd.read_parquet(os.path.join(base_dir, "output_per_esperti.parquet"))
df.shape

(3532, 20)

In [16]:
#keep only reviews (elemnts of rows) with more than 3 words

df['review_text'] = df['review_text'].apply(
    lambda reviews: [review for review in reviews if len(review.split()) > 3]
)

In [17]:
# new sorting

df_ordered = df.sort_values(
    by='review_text',
    key=lambda x: x.str.len(),  # get length of each list
    ascending=False
).reset_index(drop=True)


In [19]:
filtered_df = df[df['review_text'].apply(lambda x: 15 <= len(x) <= 20)]
filtered_df

,unique_id_user,review_text,poi_name,poi_type,poi_category,Animal Lover,Animale Spiaggia,Colto,Curioso,Disorganizzato,Esploratore,Famiglia,GenZ,Nomadi Digitali,Pigro,Romantico,Spirito Libero,Viaggiatore Affari,Viaggiatore Online,Viaggiatore Verde
57,246049,"[Pub molto caratteristico, birra ottima, Pub m...","[Roger the Club, Eurospin - Matera, Circus Caf...","[restaurants, restaurants, restaurants, restau...","[Sushi, Alimentari, Bar, Alimentari, Parco, Ri...",0.004348,0.010526,0.106522,0.295652,0.079891,0.034783,0.189674,0.267391,0.023913,0.042935,0.094565,0.173913,0.008696,0.193478,0.080435
81,423356,"[Unico per attrazioni e location, Struttura nu...","[Concattedrale di Sant'Andrea Apostolo, Parco ...","[attractions, attractions, restaurants, restau...","[Cattedrale cattolica, Parco divertimenti, Piz...",0.005263,0.000000,0.139474,0.296053,0.081579,0.052632,0.248026,0.235526,0.008553,0.026974,0.081579,0.132895,0.003947,0.180921,0.083553
161,58534,"[Buon luogo per passare una bella serata, Bel ...","[Lido La Playa Policoro, Castello società coop...","[hotel, attractions, attractions, restaurants,...","[Stabilimento balneare, Attrazione turistica, ...",0.005882,0.028889,0.049020,0.200000,0.111765,0.031373,0.219608,0.247059,0.013725,0.027451,0.064706,0.166667,0.019608,0.166667,0.050980
195,170324,"[""Rocchetta Sanseverino""\nNel comune non sono ...","[Rocchetta Sanseverino, Ristorante Pizzeria 4 ...","[attractions, restaurants, restaurants, restau...","[Fortezza, Pizza, Pizza, Ristorante, Bar caffè...",0.002273,0.021951,0.056818,0.213636,0.118182,0.045455,0.226136,0.268182,0.015909,0.009091,0.075000,0.193182,0.006818,0.162500,0.072727
223,426949,"[Ottimo parco se fosse tenuto pulito, Ottimo r...","[Villa Diamante, Parco del Fiore Bianco (ex Pa...","[restaurants, attractions, restaurants, restau...","[Italiana, Parco, Fast food, Pizza, Pizza, Ris...",0.017949,0.000000,0.079487,0.174359,0.088462,0.117949,0.276923,0.197436,0.008974,0.016667,0.088462,0.089744,0.010256,0.139744,0.107692
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605,425302,[Personale gentile e disponibile e prezzi comp...,"[Alimentari di Colangelo Domenico, Il Caminett...","[restaurants, restaurants, restaurants, restau...","[Alimentari, Pasticceria, Bar, Ristorante, Ris...",0.000000,0.000000,0.020000,0.213333,0.060000,0.006667,0.223333,0.180000,0.006667,0.073333,0.040000,0.086667,0.000000,0.160000,0.030000
606,174180,"[Prezzi modici, qualità buona personale dispon...","[Ristorante Pizzeria Locanda Da Federico, Hote...","[restaurants, hotel, restaurants, restaurants,...","[Pizza, Hotel, Pizza, Ristorante, Cinema, Pizz...",0.000000,0.006667,0.033333,0.206667,0.053333,0.040000,0.246667,0.180000,0.000000,0.033333,0.073333,0.100000,0.000000,0.126667,0.060000
607,303783,[I titolari sono molto disponibili verso i cli...,"[Tabaccheria Caffetteria Tre Torri, MD, Chiesa...","[restaurants, restaurants, attractions, restau...","[Tabaccheria, Fornaio, Chiesa, Casalinghi, Ali...",0.000000,0.000000,0.013333,0.173333,0.020000,0.000000,0.160000,0.050000,0.000000,0.006667,0.000000,0.000000,0.033333,0.106667,0.013333
608,473946,"[Un posto sicuro dove trovarsi per un caffè, U...","[Convivium Restaurant by White Station, Conviv...","[restaurants, restaurants, restaurants, restau...","[Italiana, Italiana, Pizza, Pizza, Pizza, Pub,...",0.000000,0.000000,0.033333,0.213333,0.066667,0.020000,0.273333,0.236667,0.006667,0.040000,0.043333,0.220000,0.000000,0.113333,0.073333


In [20]:
dim = 100
final_df = filtered_df.head(dim)
final_df.to_parquet(os.path.join(base_dir, f"output_per_esperti_filtrato{dim}.parquet"), index=False)


In [21]:
test = pd.read_parquet(os.path.join(base_dir, f"output_per_esperti_filtrato{dim}.parquet"))
test.shape

(100, 20)